# Práctico 2: Manipulación de tablas y series de tiempo 

## Obtención de los datos

In [137]:
import pandas as pd
df=pd.read_csv("starbucks.csv")

### Primeros datos

In [138]:
df.head(10) #Impresión de los primeros 10 registros

,Store Number,Store Name,Address,City
0,10429-100710,Palmdale & Hwy 395,14136 US Hwy 395 Adelanto CA,Adelanto
1,635-352,Kanan & Thousand Oaks,5827 Kanan Road Agoura CA,Agoura
2,74510-27669,Vons-Agoura Hills #2001,5671 Kanan Rd. Agoura Hills CA,Agoura Hills
3,29839-255026,Target Anaheim T-0677,8148 E SANTA ANA CANYON ROAD AHAHEIM CA,AHAHEIM
4,23463-230284,Safeway - Alameda 3281,2600 5th Street Alameda CA,Alameda
5,6479-62999,Park & Central Alameda,1364 Park Street Alameda CA,Alameda
6,5535-728,Webster & Atlantic - Alameda,720 Atlantic Avenue Alameda CA,Alameda
7,74877-100291,Safeway - Alameda #2708,2227 South Shore Center Alameda CA,Alameda
8,11161-103516,Tilden & Blanding,"2671 Blanding Avene, D Alameda CA",Alameda
9,19859-196187,Target Alameda T-2829,2700 Fifth St Alameda CA,Alameda


## Operaciones con tablas

### a. Búsqueda de tienda por dirección utilizando .loc

In [139]:
df.loc[df['Address']== '1444 Shattuck Place Berkeley CA'] #Se busca en la columna Address el registro que contenga la dirección

,Store Number,Store Name,Address,City
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley


### b. Filtrado de registros por ciudad (Berkeley)

In [140]:
df.loc[df['City']== 'Berkeley'] #Se busca en la columna City los registros que contengan a Berkeley como ciudad

,Store Number,Store Name,Address,City
153,5406-945,2224 Shattuck - Berkeley,2224 Shattuck Avenue Berkeley CA,Berkeley
154,570-512,Solano Ave,1799 Solano Avenue Berkeley CA,Berkeley
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley
156,19864-202264,Telegraph & Ashby,3001 Telegraph Avenue Berkeley CA,Berkeley
157,9217-9253,2128 Oxford St.,2128 Oxford Street Berkeley CA,Berkeley


### c. Obtención de coordenadas geográficas (latitud y longitud)


In [141]:
from geopy.geocoders import Nominatim #Convertir direcciones en coordenadas
geolocator = Nominatim(user_agent="kaggle_learn") #Crea el geolocalizador que hace la busqueda
direccionTienda= df['Address'][14] #Prueba con la fila 15
locationPrueba = geolocator.geocode(direccionTienda) #Toma la dirección y devuelve un objeto(dirección, latitud, longitud)
print(locationPrueba.address)
print((locationPrueba.latitude, locationPrueba.longitude))

1220, West Main Street, Alhambra, Los Angeles County, California, 91801, United States
(34.091476, -118.1361991)


### d. Geocodificación y manejo de errores

In [142]:
import time
erroresDireccion= []

limite = 100  # límite de consultas
contador = 0

#Obtener latitud y longitud, se usa geolocalizador anterior

def obtener_coordenadas(direccion):
    global contador
    
    if contador >= limite:  # límite para no gastar consultas
        return None, None
    
    try:
        location=geolocator.geocode(direccion) #Toma la dirección y devuelve un objeto(dirección, latitud, longitud)
        time.sleep(1)  # se aumenta el tiempo para evitar saturar el servicio
        contador += 1

        if location: #Si encuentra la dirección devuelve latitud y longitud
            return location.latitude, location.longitude
        else:
            erroresDireccion.append(direccion) # Ocurre error, se agrega a lista de errores de dirección
            return None, None 

    except Exception as e:
        erroresDireccion.append(direccion)  # por si ocurre algo inesperado
        contador += 1
        return None, None
    
        
df[['Latitude', 'Longitude']] = df['Address'].apply( lambda x: pd.Series(obtener_coordenadas(x))) #Para la columna address se crean dos nuevas columnas (Latitud y Longitud) y para cada fila se llama a la función

#Mostramos resultados
print(df.head())

print("\nDirecciones con error:")
print(erroresDireccion)
print(f"\nDe las {contador} filas leídas, tenían errores:", len(erroresDireccion))


   Store Number               Store Name  \
0  10429-100710       Palmdale & Hwy 395   
1       635-352    Kanan & Thousand Oaks   
2   74510-27669  Vons-Agoura Hills #2001   
3  29839-255026    Target Anaheim T-0677   
4  23463-230284   Safeway - Alameda 3281   

                                   Address          City   Latitude  \
0             14136 US Hwy 395 Adelanto CA      Adelanto  34.506998   
1                5827 Kanan Road Agoura CA        Agoura  34.155889   
2           5671 Kanan Rd. Agoura Hills CA  Agoura Hills  34.153550   
3  8148 E SANTA ANA CANYON ROAD AHAHEIM CA       AHAHEIM        NaN   
4               2600 5th Street Alameda CA       Alameda  37.785531   

    Longitude  
0 -117.399952  
1 -118.756569  
2 -118.759190  
3         NaN  
4 -122.280575  

Direcciones con error:
['8148 E SANTA ANA CANYON ROAD AHAHEIM CA', '2671 Blanding Avene, D Alameda CA', '2210-J South Shore Drive Alameda CA', '2400 W. Commonwealth Alhambra CA', '26932 La Paz Ave Aliso Viejo CA

### e. Corrección de direcciones con fallos de geocodificación

In [ ]:
import re
import time

corregidas = []
fallidas = []

# -------------------------
# FUNCIÓN DE LIMPIEZA
# -------------------------
def limpiar_direccion(direccion):
    d = direccion.lower()

    # Errores de tipeo identificados
    d = re.sub(r'ahaheim', 'anaheim', d)
    d = re.sub(r'avene\b', 'avenue', d)

    # Guiones con letra (2210-J → 2210)
    d = re.sub(r'(\d+)-[a-z]\b', r'\1', d)

    # Eliminar todo después de la segunda coma 
    partes_coma = d.split(',')
    if len(partes_coma) >= 3:
        d = ','.join(partes_coma[:2])
    elif len(partes_coma) == 2:
        segunda = partes_coma[1].strip()
        if not re.match(r'^\d', segunda) and len(segunda.split()) > 2:
            d = partes_coma[0]

    # Números de suite/apt después de coma al final (", 101" → "")
    d = re.sub(r',\s*\d+\s*$', '', d)
    # Números de suite/apt en medio (", 101,") → ","
    d = re.sub(r',\s*\d{1,4}\s*,', ',', d)

    # Abreviaturas de tipo de calle
    d = re.sub(r'\bhwy\.?\b',      'highway', d)
    d = re.sub(r'\brd\.?\b',       'road',    d)
    d = re.sub(r'\bave\.?\b',      'avenue',  d)
    d = re.sub(r'\bblvd\.?\b',     'boulevard', d)
    d = re.sub(r'\bst\.?\b',       'street',  d)
    d = re.sub(r'\bdr\.?\b',       'drive',   d)
    d = re.sub(r'\bln\.?\b',       'lane',    d)
    d = re.sub(r'\bct\.?\b',       'court',   d)
    d = re.sub(r'\bpkwy\.?\b',     'parkway', d)
    d = re.sub(r'\bfwy\.?\b',      'freeway', d)

    # Eliminar info secundaria (suite, unit, etc.)
    d = re.sub(r'\b(suite|ste|unit|bldg|building|space|apt|floor|fl)\s*#?\w*', '', d)
    d = re.sub(r'#\d+', '', d)

    # Eliminar nombre propio + tipo comercial (ej: "Rock Creek Plaza", "Five Cities Center")
    if re.match(r'^\d+', d):
        d = re.sub(r'\b\w+\s+(plaza|mall|marketplace|shopping\s*center|towne?\s*center)\b', '', d)

    # Remover tipos comerciales sueltos que quedaron
    d = re.sub(r'\b(mall|plaza|marketplace|shopping\s*center|towne\s*center|town\s*center|center|junction|napa\s*junction)\b', '', d)

    # Quitar letras sueltas (unidades tipo "D", "J", etc.)
    d = re.sub(r'\b[a-z]\b', '', d)

    # Quitar caracteres especiales y espacios extra
    d = re.sub(r'[^a-z0-9\s]', ' ', d)
    d = re.sub(r'\s+', ' ', d).strip()

    # Ciudades pequeñas que geopy no resuelve bien → agregar condado
    ciudades_dificiles = {
        'aliso viejo':      'aliso viejo orange county california',
        'american canyon':  'american canyon napa county california',
        'alpine':           'alpine san diego county california',
        'anderson':         'anderson shasta county california',
    }
    for ciudad, reemplazo in ciudades_dificiles.items():
        if ciudad in d:
            d = re.sub(ciudad + r'.*$', reemplazo, d)
            return d  # ya tiene estado completo

    # Asegurar que termine con estado
    if not d.endswith(' ca') and ' ca ' not in d and 'california' not in d:
        d += ' ca'

    return d



# FUNCIÓN DE GEOCODIFICACIÓN CON FALLBACK

def geocodificar_con_fallback(geolocator, direccion_limpia):
    """
    Intenta geocodificar con versiones progresivamente más simples.
    Devuelve (location, intento_usado) o (None, direccion_limpia).
    """
    intentos = [direccion_limpia]

    partes = direccion_limpia.split()

    # Intento 2: número + 2 palabras de calle + últimas 2 (ciudad + estado)
    if len(partes) >= 5:
        simplificada = ' '.join(partes[:3] + partes[-2:])
        intentos.append(simplificada)

    # Intento 3: sin número de calle (solo calle + ciudad + estado)
    sin_numero = re.sub(r'^\d+\s+', '', direccion_limpia)
    if sin_numero != direccion_limpia:
        intentos.append(sin_numero)

    for intento in intentos:
        try:
            location = geolocator.geocode(intento)
            time.sleep(1)
            if location:
                return location, intento
        except Exception:
            time.sleep(1)

    return None, direccion_limpia


# -------------------------
# LOOP PRINCIPAL
# -------------------------
print("\nReintentando direcciones con limpieza (punto e)...\n")

for direccion in erroresDireccion:

    direccion_limpia = limpiar_direccion(direccion)
    location, intento_usado = geocodificar_con_fallback(geolocator, direccion_limpia)

    if location:
        corregidas.append({
            "original":   direccion,
            "modificada": intento_usado,
            "lat":        location.latitude,
            "lon":        location.longitude
        })
    else:
        fallidas.append({
            "original":   direccion,
            "modificada": direccion_limpia
        })


# -------------------------
# RESULTADOS
# -------------------------

print("RESULTADOS FINALES ")


print("\n Corregidas:")
for c in corregidas:
    print(c)

print("\n Fallidas:")
for i, f in enumerate(fallidas, 1):
    print(f"\n[{i}]")
    print("  Original  :", f['original'])
    print("  Modificada:", f['modificada'])

print("\n RESUMEN:")
print("Corregidas:", len(corregidas))
print("Fallidas:",   len(fallidas))